In [20]:
# %%
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
# %% JSON-Export-Helper (MAE/RMSE gesamt + pro Schritt)
import os, json
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def export_metrics_json(*,
                        model: str,
                        resolution: str,          # "minute" oder "hour"
                        input_width: int,
                        horizon: int,
                        feature_columns,
                        y_true: np.ndarray,
                        y_pred: np.ndarray,
                        out_dir: str = "results"):
    # Gesamtmetriken
    mae_total = float(mean_absolute_error(y_true, y_pred))
    rmse_total = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    # Pro-Schritt-Metriken (Länge = horizon)
    mae_per_step = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
    rmse_per_step = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='raw_values'))

    payload = {
        "model": model,
        "resolution": resolution,
        "input_width": int(input_width),
        "horizon": int(horizon),
        "features": list(feature_columns),
        "metrics_total": {
            "MAE": mae_total,
            "RMSE": rmse_total
        },
        "metrics_per_step": {
            "MAE": np.asarray(mae_per_step).tolist(),
            "RMSE": np.asarray(rmse_per_step).tolist()
        }
    }

    os.makedirs(out_dir, exist_ok=True)
    fname = f"metrics_{model}_{resolution}_in{input_width}_out{horizon}.json"
    fpath = os.path.join(out_dir, fname)
    with open(fpath, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"✅  Saved: {fpath}")


In [3]:
# %%
train = pd.read_csv('household_train.csv', parse_dates=['Datetime'])
valid = pd.read_csv('household_validation.csv', parse_dates=['Datetime'])
test  = pd.read_csv('household_test.csv',  parse_dates=['Datetime'])

# %%
train.set_index('Datetime', inplace=True)
valid.set_index('Datetime', inplace=True)
test.set_index('Datetime',  inplace=True)


In [4]:
# %%
numeric_features = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]

# Modell-Input-Features (wie in deinem Ridge-Skript verwendet)
feature_columns = [
    "Global_active_power", "Global_intensity",
    "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]

# %%
train_hourly = train[numeric_features].resample('H').mean()
valid_hourly = valid[numeric_features].resample('H').mean()
test_hourly  = test[numeric_features].resample('H').mean()


In [5]:
# %%
# 30 Minuten (minütliche Daten)
scaler_min = StandardScaler()
train_min_scaled = pd.DataFrame(
    scaler_min.fit_transform(train[feature_columns]),
    index=train.index, columns=feature_columns
)
valid_min_scaled = pd.DataFrame(
    scaler_min.transform(valid[feature_columns]),
    index=valid.index, columns=feature_columns
)
test_min_scaled = pd.DataFrame(
    scaler_min.transform(test[feature_columns]),
    index=test.index, columns=feature_columns
)

# 24 Stunden (stündliche Daten)
scaler_hr = StandardScaler()
train_hr_scaled = pd.DataFrame(
    scaler_hr.fit_transform(train_hourly[feature_columns]),
    index=train_hourly.index, columns=feature_columns
)
valid_hr_scaled = pd.DataFrame(
    scaler_hr.transform(valid_hourly[feature_columns]),
    index=valid_hourly.index, columns=feature_columns
)
test_hr_scaled = pd.DataFrame(
    scaler_hr.transform(test_hourly[feature_columns]),
    index=test_hourly.index, columns=feature_columns
)


## Windowing

In [6]:
# %%
def create_sequences(data, input_width, label_width, feature_columns, target_column="Global_active_power"):
    X, y = [], []
    values_X = data[feature_columns].values
    values_y = data[target_column].values
    for i in range(len(data) - input_width - label_width):
        X.append(values_X[i:i+input_width])
        y.append(values_y[i+input_width:i+input_width+label_width])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


In [7]:
# %%
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1e-8))) * 100.0

def eval_multi_step(y_true, y_pred, set_name=""):
    mae_per_step = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
    mae_total = mean_absolute_error(y_true, y_pred)
    rmse_total = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_total = mean_absolute_percentage_error(y_true, y_pred)
    r2_total = r2_score(y_true, y_pred)
    print(f"\n{set_name} — Gesamt: MAE={mae_total:.4f}, RMSE={rmse_total:.4f}, MAPE={mape_total:.2f}%, R²={r2_total:.4f}")
    return {
        "MAE": mae_total,
        "RMSE": rmse_total,
        "MAPE": mape_total,
        "R2": r2_total,
        "MAE_per_step": mae_per_step
    }


## 30-Minuten Vorhersage

In [8]:
# %%
input_width_30 = 30   # entspricht n_lags in Ridge (30 Minuten)
label_width_30 = 30   # entspricht forecast_horizon in Ridge

X_train_30, y_train_30 = create_sequences(train_min_scaled, input_width_30, label_width_30, feature_columns)
X_valid_30, y_valid_30 = create_sequences(valid_min_scaled, input_width_30, label_width_30, feature_columns)
X_test_30,  y_test_30  = create_sequences(test_min_scaled,  input_width_30, label_width_30, feature_columns)


In [ ]:
# %%
early_stop = EarlyStopping(
    monitor="val_loss",       # oder "val_mae", wenn dir MAE wichtiger ist
    patience=10,              # wie viele Epochen ohne Verbesserung abgewartet wird
    restore_best_weights=True # nach Abbruch beste Gewichte wiederherstellen
)

model_30 = Sequential([
    Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(input_width_30, len(feature_columns))),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(label_width_30)
])

model_30.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
                 loss='mse', metrics=['mae'])

history_30 = model_30.fit(
    X_train_30, y_train_30,
    validation_data=(X_valid_30, y_valid_30),
    epochs=10,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop]
)

test_loss_30, test_mae_30 = model_30.evaluate(X_test_30, y_test_30, verbose=0)
y_pred_30 = model_30.predict(X_test_30, verbose=0)
metrics_30 = eval_multi_step(y_test_30, y_pred_30, set_name="CNN 30min (Test)")


/Users/basti/miniforge3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
50006/50006 ━━━━━━━━━━━━━━━━━━━━ 163s 3ms/step - loss: 0.3475 - mae: 0.3308 - val_loss: 0.3004 - val_mae: 0.3196
Epoch 2/10
50006/50006 ━━━━━━━━━━━━━━━━━━━━ 166s 3ms/step - loss: 0.3305 - mae: 0.3195 - val_loss: 0.2944 - val_mae: 0.3041
Epoch 3/10
50006/50006 ━━━━━━━━━━━━━━━━━━━━ 167s 3ms/step - loss: 0.3257 - mae: 0.3164 - val_loss: 0.2953 - val_mae: 0.3191
Epoch 4/10
50006/50006 ━━━━━━━━━━━━━━━━━━━━ 122s 2ms/step - loss: 0.3231 - mae: 0.3147 - val_loss: 0.2908 - val_mae: 0.3107
Epoch 5/10
50006/50006 ━━━━━━━━━━━━━━━━━━━━ 131s 3ms/step - loss: 0.3213 - mae: 0.3138 - val_loss: 0.2934 - val_mae: 0.3016
Epoch 6/10
50006/50006 ━━━━━━━━━━━━━━━━━━━━ 142s 3ms/step - loss: 0.3198 - mae: 0.3128 - val_loss: 0.2922 - val_mae: 0.3052
Epoch 7/10
50006/50006 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - loss: 0.3187 - mae: 0.3124 - val_loss: 0.2893 - val_mae: 0.2963
Epoch 8/10
50006/50006 ━━━━━━━━━━━━━━━━━━━━ 157s 3ms/step - loss: 0.3177 - mae: 0.3119 - val_loss: 0.2900 - val_mae: 0.3042
Epoch 9/

In [13]:
# %% Export JSON (CNN, 30min)
export_metrics_json(
    model="cnn",
    resolution="minute",
    input_width=input_width_30,      # = 30
    horizon=label_width_30,          # = 30
    feature_columns=feature_columns, # ["Global_active_power", ...]
    y_true=y_test_30,
    y_pred=y_pred_30,
    out_dir="results"
)


✅  Saved: results/metrics_cnn_minute_in30_out30.json


## 24h Vorhersage

In [10]:
# %%
input_width_24 = 48   # entspricht n_lags in Ridge (48 Stunden)
label_width_24 = 24   # entspricht forecast_horizon in Ridge

X_train_24, y_train_24 = create_sequences(train_hr_scaled, input_width_24, label_width_24, feature_columns)
X_valid_24, y_valid_24 = create_sequences(valid_hr_scaled, input_width_24, label_width_24, feature_columns)
X_test_24,  y_test_24  = create_sequences(test_hr_scaled,  input_width_24, label_width_24, feature_columns)


In [ ]:
# %%
early_stop = EarlyStopping(
    monitor="val_loss",       # oder "val_mae", wenn dir MAE wichtiger ist
    patience=10,              # wie viele Epochen ohne Verbesserung abgewartet wird
    restore_best_weights=True # nach Abbruch beste Gewichte wiederherstellen
)

model_24 = Sequential([
    Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(input_width_24, len(feature_columns))),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(label_width_24)
])

model_24.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
                 loss='mse', metrics=['mae'])

history_24 = model_24.fit(
    X_train_24, y_train_24,
    validation_data=(X_valid_24, y_valid_24),
    epochs=4000,
    batch_size=64,
    verbose=1,
    callbacks=[early_stop]
)

test_loss_24, test_mae_24 = model_24.evaluate(X_test_24, y_test_24, verbose=0)
y_pred_24 = model_24.predict(X_test_24, verbose=0)
metrics_24 = eval_multi_step(y_test_24, y_pred_24, set_name="CNN 24h (Test)")


Epoch 1/4000
416/416 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.7138 - mae: 0.6274 - val_loss: 0.5750 - val_mae: 0.5621
Epoch 2/4000
416/416 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.6318 - mae: 0.5801 - val_loss: 0.5807 - val_mae: 0.5586
Epoch 3/4000
416/416 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6110 - mae: 0.5679 - val_loss: 0.5686 - val_mae: 0.5536
Epoch 4/4000
416/416 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5976 - mae: 0.5611 - val_loss: 0.5663 - val_mae: 0.5536
Epoch 5/4000
416/416 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.5863 - mae: 0.5547 - val_loss: 0.5629 - val_mae: 0.5530
Epoch 6/4000
416/416 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.5765 - mae: 0.5497 - val_loss: 0.5786 - val_mae: 0.5561
Epoch 7/4000
416/416 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.5676 - mae: 0.5452 - val_loss: 0.5697 - val_mae: 0.5537
Epoch 8/4000
416/416 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5589 - mae: 0.5408 - val_loss: 0.5744 - val_mae: 0.5570
Epoch 9/4000
416/416 ━━━━━━━━━━━━━━━━━━━

In [16]:
# %% Export JSON (CNN, 24h)
export_metrics_json(
    model="cnn",
    resolution="hour",
    input_width=input_width_24,      # = 48
    horizon=label_width_24,          # = 24
    feature_columns=feature_columns,
    y_true=y_test_24,
    y_pred=y_pred_24,
    out_dir="results"
)


✅  Saved: results/metrics_cnn_hour_in48_out24.json


In [12]:
# %%
print("\n================ VERGLEICH CNN 30min vs. CNN 24h ================")
print(f"30min — MAE:  {metrics_30['MAE']:.4f} | RMSE: {metrics_30['RMSE']:.4f} | MAPE: {metrics_30['MAPE']:.2f}% | R²: {metrics_30['R2']:.4f}")
print(f"24h   — MAE:  {metrics_24['MAE']:.4f} | RMSE: {metrics_24['RMSE']:.4f} | MAPE: {metrics_24['MAPE']:.2f}% | R²: {metrics_24['R2']:.4f}")



================ VERGLEICH CNN 30min vs. CNN 24h ================
30min — MAE:  0.2797 | RMSE: 0.4760 | MAPE: 1276641800.00% | R²: 0.6298
24h   — MAE:  0.4848 | RMSE: 0.6530 | MAPE: 2616665200.00% | R²: 0.2570
